In [ ]:
!pip -q install gradio

In [ ]:
import re
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

ROLL_NUMBER = 1024170178
queries_by_intent = {
    "heart_basics": [
        "What does the heart do?", "How does blood move through the heart?",
        "What is the function of heart valves?", "What is a normal resting heart rate?",
        "Explain the chambers of the heart.", "What is the circulatory system?",
        "How does the heart pump blood?", "What is the role of arteries?",
        "What are veins?", "Why is the heart important?"
    ],
    "risk_prevention": [
        "How can I reduce heart disease risk?", "Does walking help heart health?",
        "What foods support a healthy heart?", "How does smoking affect the heart?",
        "Can exercise lower blood pressure?", "Why should I limit salt?",
        "How does sleep affect cardiovascular health?", "What is a healthy lifestyle for the heart?",
        "Does stress affect the heart?", "How can I maintain a healthy weight?"
    ],
    "conditions_symptoms": [
        "What is high blood pressure?", "What is coronary artery disease?",
        "What is cholesterol?", "What are common signs of heart disease?",
        "What is an irregular heartbeat?", "What is heart failure?",
        "What does angina mean?", "What is a heart murmur?",
        "What is atherosclerosis?", "What is a heart attack?"
    ],
    "tests_monitoring": [
        "What is an ECG test?", "Why is blood pressure measured?",
        "What does a cholesterol test show?", "What is an echocardiogram?",
        "How does a stress test work?", "What is a pulse oximeter?",
        "What does an LDL result mean?", "What is a cardiac blood test?",
        "How often should blood pressure be checked?", "What is a heart rate monitor?"
    ],
    "medication_care": [
        "Why might a doctor prescribe statins?", "What are beta blockers used for?",
        "Should I stop my heart medicine?", "Why is medication adherence important?",
        "What should I ask my doctor about aspirin?", "Can medicines interact?",
        "How should I discuss side effects with my doctor?", "What is a blood thinner?",
        "Can I change my prescribed dose?", "Why are follow up appointments important?"
    ]
}
corpus = pd.DataFrame([{"query": q, "intent": intent} for intent, qs in queries_by_intent.items() for q in qs])
print(corpus)

In [ ]:
taxonomy = {
    "Heart foundations": ["Heart structure", "Blood circulation"],
    "Prevention and lifestyle": ["Nutrition and activity", "Risk factors"],
    "Conditions and symptoms": ["Cardiovascular conditions", "Warning signs"],
    "Testing and monitoring": ["Diagnostic tests", "Vital measurements"],
    "Treatment and care": ["Medication information", "Clinical follow-up"]
}
for major, subcategories in taxonomy.items():
    print(major, subcategories)

concepts = {
    "Heart": "Muscular organ that pumps blood.",
    "Arteries": "Vessels carrying blood away from the heart.",
    "Veins": "Vessels returning blood toward the heart.",
    "Blood pressure": "Force of blood against artery walls.",
    "Cholesterol": "A lipid measured in cardiovascular risk assessment.",
    "Plaque": "Build-up in artery walls.",
    "ECG": "Test recording electrical activity of the heart.",
    "Statins": "Medicines used to lower cholesterol.",
    "Exercise": "Physical activity supporting cardiovascular fitness.",
    "Smoking": "A cardiovascular risk factor."
}
relationships = [
    ("Heart", "pumps blood through", "Arteries"),
    ("Veins", "return blood to", "Heart"),
    ("Blood pressure", "is measured in", "Arteries"),
    ("Cholesterol", "can contribute to", "Plaque"),
    ("Plaque", "can narrow", "Arteries"),
    ("ECG", "records activity of", "Heart"),
    ("Statins", "may lower", "Cholesterol"),
    ("Exercise", "supports", "Heart"),
    ("Smoking", "increases risk of", "Heart disease"),
    ("High blood pressure", "is a risk factor for", "Heart disease"),
    ("Doctor", "interprets", "ECG"),
    ("Patient", "discusses medicines with", "Doctor")
]
print(concepts)
print(relationships)

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = re.findall(r"\b[a-z]+\b", text.lower())
    tokens = [t for t in tokens if t not in stop_words]
    return " ".join(lemmatizer.lemmatize(t) for t in tokens)

X_train, X_test, y_train, y_test = train_test_split(
    corpus["query"], corpus["intent"], test_size=0.30,
    random_state=42, stratify=corpus["intent"]
)
model = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=preprocess, ngram_range=(1, 2))),
    ("classifier", MultinomialNB())
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, predictions, average="weighted", zero_division=0)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print(classification_report(y_test, predictions, zero_division=0))
labels = sorted(corpus["intent"].unique())
cm = confusion_matrix(y_test, predictions, labels=labels)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(xticks_rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
knowledge_base = {
    "heart_basics": "The heart is a muscular organ that circulates blood. It has four chambers and valves that help blood flow in the correct direction.",
    "risk_prevention": "Heart-health habits can include regular physical activity, balanced eating, avoiding tobacco, adequate sleep, and discussing personal risk factors with a healthcare professional.",
    "conditions_symptoms": "Cardiovascular conditions include high blood pressure, coronary artery disease, rhythm disorders, and heart failure. A healthcare professional should assess health concerns.",
    "tests_monitoring": "Common assessments include blood pressure measurement, cholesterol testing, ECG, and echocardiography. A clinician explains and interprets appropriate tests.",
    "medication_care": "Heart medicines are prescribed for individual reasons. Do not start, stop, or change a medicine or dose without consulting the prescribing healthcare professional."
}

EMERGENCY_TERMS = [
    "chest pain", "chest pressure", "can't breathe", "cannot breathe",
    "severe shortness of breath", "fainting", "unconscious",
    "face drooping", "one-sided weakness", "sudden weakness"
]

def is_emergency(query):
    q = query.lower()
    return any(term in q for term in EMERGENCY_TERMS)

def assistant_response(query, threshold=0.25):
    if is_emergency(query):
        return "This may be a medical emergency. Call your local emergency number now (112 in India) or go to the nearest emergency department. Do not drive yourself. Follow emergency professionals' instructions."
    probabilities = model.predict_proba([query])[0]
    best_index = int(np.argmax(probabilities))
    confidence = float(probabilities[best_index])
    intent = model.classes_[best_index]
    if confidence < threshold:
        return "I am not confident that I understood. Please rephrase or ask about heart basics, prevention, conditions, tests, or medication information."
    if intent not in knowledge_base:
        return "I can only provide general heart-health information within the supported topics."
    return f"Detected intent: {intent} (confidence {confidence:.2f}).\n{knowledge_base[intent]}\nThis is general education, not a diagnosis or personal medical advice."

def query_assistant(query):
    if not query.strip():
        return "Please enter a question."
    return assistant_response(query)

test_queries = [
    "What does the heart do?",
    "How can I prevent heart disease?",
    "What is high blood pressure?",
    "What is an ECG?",
    "Should I stop taking my prescribed medicine?",
    "My chest hurts and I cannot breathe",
    "Tell me about treating a broken ankle",
    "blue clouds running quickly"
]
for query in test_queries:
    print(query)
    print(query_assistant(query))

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=query_assistant,
    inputs=gr.Textbox(lines=3, label="Ask a heart-health question"),
    outputs=gr.Textbox(lines=6, label="Assistant response"),
    title="Heart Health Information Assistant",
    description="Educational information only. For possible emergencies, call 112 in India."
)
demo.launch(share=True)